## Strategy Feature Reference

Each trading day x battery combination is represented by 15 features for UMAP + clustering, using FEATURE_COLUMNS in nem_battery/strategy_map.py. Features are z-score scaled with StandardScaler before UMAP.

---

### Features Used By UMAP

| Category | Feature | Description | Formula (implemented) |
|---|---|---|---|
| Operational complexity | state_reversal_count | Number of charge/discharge/idle state changes in the day (using sign of net MW) | sum(diff(sign(energy_mw)) != 0) |
| Operational complexity | normalised_total_variation | Choppiness of net MW trajectory | sum(abs(diff(energy_mw))) / (sum(abs(energy_mw)) + epsilon) |
| Operational complexity | utilization_factor | Share of intervals with non-zero dispatch | mean((discharge_mw > 0) or (charge_mw > 0)) |
| Market reactivity | energy_price_pearson_correlation | Linear relationship between net MW and price | corr(rrp, energy_mw) |
| Market reactivity | energy_price_spearman_correlation | Rank-order relationship between net MW and price | corr(rank(rrp), rank(energy_mw)) |
| Market reactivity | price_selectivity_index | Spread between prices when net exporting vs net importing, normalized to [-1, 1] | (mean(rrp | energy_mw > 0) - mean(rrp | energy_mw < 0)) / (abs(mean_pos) + abs(mean_neg)) |
| Market reactivity | negative_price_capture | Average charging MW during negative-price intervals | mean(charge_mw | rrp < 0) |
| Value stacking | fcas_revenue_share | FCAS share of total value stack (energy value + FCAS) | total_fcas / (abs(energy_revenue - energy_cost) + total_fcas) |
| Value stacking | reg_vs_contingency_ratio | Regulation FCAS contribution relative to total FCAS | (raisereg + lowerreg) / total_fcas |
| Value stacking | revenue_diversity_index | Shannon entropy across 5 revenue streams | -sum(p_i * ln(p_i)) over [energy_value, raise_reg, lower_reg, raise_cont, lower_cont] |
| Value stacking | co_optimization_frequency | Share of intervals with simultaneous energy dispatch and FCAS | mean((abs(energy_mw) > 0) and (total_fcas_interval > 0)) |
| Temporal strategy | evening_peak_weight | Fraction of daily discharge in evening peak window | sum(discharge_mw in 17:00-21:00) / sum(discharge_mw) |
| Temporal strategy | morning_peak_weight | Fraction of daily discharge in morning peak window | sum(discharge_mw in 06:00-09:00) / sum(discharge_mw) |
| Temporal strategy | solar_soak_charge_weight | Fraction of daily charge in solar soak window | sum(charge_mw in 10:00-15:00) / sum(charge_mw) |
| Temporal strategy | overnight_charge_weight | Fraction of daily charge in overnight window | sum(charge_mw in 00:00-04:00) / sum(charge_mw) |

---

* there were other features considered but this is the set of features that provided a good separation between clusters, where each cluster can have distinct characteristics. 

### Pre-processing Notes (as implemented)

- Missing price values are imputed in sequence: interval-region mean, then region mean, then global mean.
- Missing MW and FCAS/revenue components are set to 0.
- total_fcas, net, and energy_mw are recomputed from component columns.
- Safe division logic is used to avoid divide-by-zero blowups (epsilon threshold).
- Days with outlier values of the rrp are removed. This is done to remove high revenue days that are just due to external events triggering a spike in the price of energy. 
- Zero revenue days are removed. 


In [ ]:
from nem_battery.strategy_map import (
    load_interval_data,
    prepare_interval_data,
    build_feature_frame,
    train_strategy_model,
    FEATURE_COLUMNS,
    EPSILON,
    plot_cluster_battery_count,
    StrategyArtifacts,
    plot_cluster_feature_means,
    plot_cluster_revenue,
    plot_opportunity_capture,
)

from nem_battery.battery import KNOWN_BATTERIES
import plotly.express as px
import duckdb
from typing import Literal

mwh_capacity_map = {v.name: v.mwh_capacity for k, v in KNOWN_BATTERIES.items()}

{'Hornsdale Power Reserve': 193.5,
 'Victorian Big Battery': 450.0,
 'Wallgrove BESS': 75.0,
 'Lake Bonney BESS': 52.0,
 'Gannawarra ESS': 50.0,
 'Dalrymple North BESS': 8.0,
 'Wandoan Power BESS': 150.0,
 'Torrens Island BESS': 250.0,
 'Blyth BESS': 477.0,
 'Templers BESS': 330.0,
 'Capital Battery': 200.0,
 'Rangebank BESS': 400.0,
 'Hazelwood BESS': 150.0,
 'Koorangie BESS': 370.0,
 'Tarong BESS': 600.0,
 'Western Downs BESS': 1080.0,
 'Greenbank BESS': 400.0}

In [2]:
# generate the features dataframe
raw = load_interval_data(target="remote")
# raw = raw[raw["battery_key"] != "lake_bonney"]
prepared = prepare_interval_data(raw)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
# generate the features dataframe
# raw = load_interval_data(target="remote")
# # raw = raw[raw["battery_key"] != "lake_bonney"]
# prepared = prepare_interval_data(raw)
features = build_feature_frame(prepared)
features = features[features["net"] != 0].reset_index(drop=True)
p95_max_rrp = features["max_rrp"].quantile(0.95)
features = features[features["max_rrp"] <= p95_max_rrp].reset_index(drop=True)

In [4]:
# drop_cols = ["charge_discharge_ratio", "discharge_peak_intensity", "resting_time_avg"]
drop_cols = []

n_clusters = 3
artifacts = train_strategy_model(
    features,
    clusterer="kmeans",
    n_clusters=n_clusters,
    drop_cols=drop_cols,
)


/home/ian/nem-battery/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/ian/nem-battery/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [5]:
def plot_2d_embedding(
    artifacts: StrategyArtifacts,
    color_by: Literal[
        "cluster", "battery_key", "net_revenue", "revenue_per_mwh", "opportunity_capture"
    ] = "cluster",
    cap_outliers: bool = True,
):
    embedding_df = artifacts.embedding_2d

    continuous_cols = ["net_revenue", "revenue_per_mwh", "opportunity_capture"]
    if color_by in continuous_cols:
        cap_95 = (
            embedding_df[color_by].quantile(0.95) if cap_outliers else embedding_df[color_by].max()
        )
        p5 = (
            embedding_df[color_by].quantile(0.05) if cap_outliers else embedding_df[color_by].min()
        )
        fig = px.scatter(
            embedding_df,
            x="x",
            y="y",
            color=color_by,
            color_continuous_scale="Greens",
            range_color=(p5, cap_95),
            hover_data={"battery_name": True, "trading_day": True},
            title=f"2D Embedding of Battery Intervals, Colored by {color_by} (capped at 95th percentile)",
        )
    else:
        fig = px.scatter(
            embedding_df,
            x="x",
            y="y",
            color=color_by,
            hover_data={"battery_name": True, "trading_day": True},
            title=f"2D Embedding of Battery Intervals, Colored by {color_by}",
        )

    fig.show()


plot_2d_embedding(artifacts, color_by="battery_key")


In [6]:
plot_2d_embedding(artifacts, color_by="cluster")


In [7]:
plot_2d_embedding(artifacts, color_by="net_revenue", cap_outliers=True)

In [8]:
plot_2d_embedding(artifacts, color_by="revenue_per_mwh")


In [9]:
import ipywidgets as widgets
from IPython.display import display


def plot_2d_embedding_w_highlight(
    artifacts: StrategyArtifacts,
    highlight_battery: str,
):
    embedding_df = artifacts.embedding_2d.copy()

    base_df = embedding_df[embedding_df["battery_key"] != highlight_battery]
    highlight_df = embedding_df[embedding_df["battery_key"] == highlight_battery]

    fig = px.scatter(
        base_df,
        x="x",
        y="y",
        color="cluster",
        opacity=0.20,
        hover_data={"battery_name": True, "trading_day": True},
        title=f"2D Embedding by Cluster (highlight: {highlight_battery})",
        width=1600,
    )

    if not highlight_df.empty:
        fig.add_scatter(
            x=highlight_df["x"],
            y=highlight_df["y"],
            mode="markers",
            name=f"{highlight_battery} (highlight)",
            marker=dict(
                color="#FF2D55",
                size=6,
                opacity=1.0,
                line=dict(color="black", width=1.5),
            ),
            customdata=highlight_df[["battery_name", "trading_day"]],
            hovertemplate=(
                "battery_name=%{customdata[0]}<br>"
                "trading_day=%{customdata[1]}<br>"
                "x=%{x}<br>"
                "y=%{y}<extra></extra>"
            ),
        )

    fig.show()


battery_options = sorted(artifacts.embedding_2d["battery_key"].dropna().unique().tolist())
default_battery = "lake_bonney" if "lake_bonney" in battery_options else battery_options[0]

battery_dropdown = widgets.Dropdown(
    options=battery_options,
    value=default_battery,
    description="Battery:",
)

interactive_plot = widgets.interactive_output(
    lambda highlight_battery: plot_2d_embedding_w_highlight(artifacts, highlight_battery),
    {"highlight_battery": battery_dropdown},
)

display(battery_dropdown, interactive_plot)

Dropdown(description='Battery:', index=8, options=('blyth', 'capital_battery', 'dalrymple_north', 'gannawarra'…

Output()

In [10]:
plot_cluster_battery_count(artifacts)

In [11]:
plot_cluster_feature_means(artifacts)

In [12]:
cluster_summary = artifacts.cluster_summary.sort_values("cluster")
feature_cols = [col for col in FEATURE_COLUMNS if col in cluster_summary.columns]

cluster_feature_means = cluster_summary.set_index("cluster")[feature_cols]
relative_values = (cluster_feature_means / (cluster_feature_means.sum(axis=0) + EPSILON)).T
absolute_values = cluster_feature_means.T
print(absolute_values)

cluster                                    0          1          2
state_reversal_count               42.268912  41.961369  43.108503
normalised_total_variation          0.443265   0.541769   0.822479
utilization_factor                  0.447351   0.329430   0.247751
energy_price_pearson_correlation    0.542580   0.414815   0.338452
energy_price_spearman_correlation   0.560664   0.376279   0.259761
price_selectivity_index             0.755925   0.627428   0.593044
fcas_revenue_share                  0.164459   0.015401   0.367102
reg_vs_contingency_ratio            0.654422   0.049756   0.634963
revenue_diversity_index             0.459026   0.046821   0.845572
co_optimization_frequency           0.352495   0.024731   0.208798
evening_peak_weight                 0.726737   0.582780   0.446836
morning_peak_weight                 0.079800   0.115621   0.232655
solar_soak_charge_weight            0.647635   0.605971   0.556365
overnight_charge_weight             0.071720   0.106253   0.07

In [13]:
print(relative_values)

cluster                                   0         1         2
state_reversal_count               0.331938  0.329523  0.338531
normalised_total_variation         0.245099  0.299566  0.454782
utilization_factor                 0.436214  0.321228  0.241583
energy_price_pearson_correlation   0.418384  0.319864  0.260981
energy_price_spearman_correlation  0.468116  0.314167  0.216883
price_selectivity_index            0.382283  0.317300  0.299911
fcas_revenue_share                 0.300128  0.028107  0.669940
reg_vs_contingency_ratio           0.488323  0.037128  0.473803
revenue_diversity_index            0.339411  0.034620  0.625229
co_optimization_frequency          0.600478  0.042129  0.355689
evening_peak_weight                0.413541  0.331624  0.254267
morning_peak_weight                0.185981  0.269465  0.542223
solar_soak_charge_weight           0.357618  0.334611  0.307219
overnight_charge_weight            0.279961  0.414762  0.301373
negative_price_capture             0.354

# Analysis of Battery Strategy Clusters

## Cluster 0: “Smart Profit Maximiser”

**Strategy:**  
Actively optimises across multiple markets (energy and FCAS) to maximise total revenue, using precise, data-driven dispatch decisions.

**Key Characteristics:**

- **Strong Price Alignment:**  
  This cluster shows the highest Pearson (0.54) and Spearman (0.56) correlations, indicating that battery dispatch closely tracks market price movements. In simple terms, it consistently charges when prices are low and discharges when prices are high.

- **Selective Dispatch:**  
  Leads in the Price Selectivity Index (0.75), meaning it is highly disciplined about when it operates—focusing only on the most profitable intervals rather than running continuously.

- **Advanced Market Participation:**  
  Has the highest Co-optimisation Frequency (0.35), actively stacking value across both energy and ancillary services. A large portion of this comes from Regulation FCAS (0.65), indicating sophisticated multi-market bidding.

- **Efficient Operation:**  
  Combines the highest Utilisation Factor (0.45) with the lowest Normalised Total Variation (0.44). This suggests sustained, smooth operation rather than rapid or erratic switching—maximising revenue while minimising wear and inefficiency.

- **Time-of-Day Focus:**  
  Strongly targets the Evening Peak (0.73), when prices are typically highest, and makes extensive use of Solar Soak charging (0.65) to acquire low-cost energy during the day.

---

## Cluster 1: “Cheap Energy Buyer”

**Strategy:**  
Focuses on simple energy arbitrage, charging during reliably low or negative price periods and discharging later, with minimal involvement in ancillary services.

**Key Characteristics:**

- **Negative Price Specialist:**  
  Achieves the highest relative Negative Price Capture (0.36), indicating a strong focus on charging during extreme low or negative price events, even if overall price tracking is less precise.

- **Single-Market Focus:**  
  Has very low FCAS Revenue Share (0.015) and Revenue Diversity Index (0.04), showing that revenue is almost entirely driven by energy trading rather than service stacking.

- **Predictable Charging Behaviour:**  
  Displays the highest Overnight Charge Weight (0.106) along with strong Solar Soak participation (0.60). This suggests a reliance on consistent, low-cost charging windows rather than dynamic, real-time optimisation.

- **Simpler Execution Style:**  
  Moderate Utilisation and Normalised Total Variation indicate less frequent dispatch changes, consistent with a more static or rule-based bidding strategy.

---

## Cluster 2: “Grid Stabiliser”

**Strategy:**  
Prioritises providing grid support services (FCAS) over capturing energy price arbitrage, often reserving capacity to respond to system needs.

**Key Characteristics:**

- **Highly Responsive Operation:**  
  Exhibits the highest Normalised Total Variation (0.82) and State Reversal Count (43.1), reflecting rapid and frequent output changes required to follow Automatic Generation Control (AGC) signals.

- **FCAS-Driven Revenue Model:**  
  Has the highest FCAS Revenue Share (0.37) and Revenue Diversity Index (0.84), indicating strong participation across multiple ancillary service markets.

- **Weaker Energy Price Tracking:**  
  Shows lower Spearman Correlation (0.26) and Price Selectivity (0.59), as maintaining availability for FCAS often takes priority over responding to energy price signals.

- **Morning-Oriented Activity:**  
  More active during the Morning Peak (0.23) compared to other clusters, with less emphasis on the Evening Peak.

- **Reserved Capacity Profile:**  
  Lowest Utilisation Factor (0.24), suggesting the battery frequently operates in a partially idle or “on standby” state to ensure it can deliver high-value contingency services when needed.